In [0]:
# 05_gold_forecast.py
# ============================================================
# Camada Gold — Forecast por Loja x Cluster
# Modelo escolhido: Holt-Winters (validado estatisticamente
# como superior ao XGBoost no MVP2, Seção 9 — Wilcoxon p<0.001
# em amostra ampliada de 365 pares por cluster).
# Aqui comparamos apenas Naive Sazonal (baseline) vs. HW,
# reforçando a escolha já validada, sem reabrir o debate de ML.
# ============================================================

# Caso o cluster serverless não tenha statsmodels pré-instalado:
%pip install statsmodels

import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pyspark.sql import functions as F

HORIZONTE = 12

def mape(real, pred):
    mask = real != 0
    return np.mean(np.abs((real[mask] - pred[mask]) / real[mask])) * 100

def baseline_naive_sazonal(train, steps):
    ciclo = min(steps, len(train))
    return np.array(train[-ciclo:].values[:steps], dtype=float)

def ajustar_hw(train, steps):
    try:
        fit = ExponentialSmoothing(
            train, trend='add', seasonal='add',
            seasonal_periods=12, initialization_method='estimated'
        ).fit(optimized=True, remove_bias=True)
        return np.maximum(fit.forecast(steps), 0)
    except Exception:
        return np.full(steps, np.nan)

# ── Monta a série agregada por loja + cluster ──────────────
fato   = spark.table("gold.sop_forecast.fato_vendas_mensal")
bridge = spark.table("gold.sop_forecast.dim_item_loja").select("loja_id", "item_id", "cluster_id")

serie_cluster = (
    fato.join(bridge, on=["loja_id", "item_id"])
    .groupBy("loja_id", "cluster_id", "ano_mes")
    .agg(F.sum("vendas_mes").alias("vendas_cluster"))
    .toPandas()
)

registros = []
for (loja, cluster), grupo in serie_cluster.groupby(["loja_id", "cluster_id"]):
    grupo = grupo.sort_values("ano_mes")
    serie = pd.Series(grupo["vendas_cluster"].values, index=pd.to_datetime(grupo["ano_mes"]))

    if len(serie) < 24:  # precisa de pelo menos 2 ciclos sazonais
        continue

    train, test = serie.iloc[:-HORIZONTE], serie.iloc[-HORIZONTE:]

    pred_naive = baseline_naive_sazonal(train, HORIZONTE)
    pred_hw    = ajustar_hw(train, HORIZONTE)

    mape_naive = round(mape(test.values, pred_naive), 2)
    mape_hw    = round(mape(test.values, pred_hw), 2) if not np.isnan(pred_hw).any() else None

    vencedor = "Holt-Winters" if (mape_hw is not None and mape_hw < mape_naive) else "Naive Sazonal"

    for i, data_ref in enumerate(test.index):
        registros.append({
            "loja_id": int(loja), "cluster_id": int(cluster),
            "ano_mes": data_ref, "real": float(test.values[i]),
            "pred_naive": float(pred_naive[i]),
            "pred_hw": float(pred_hw[i]) if pred_hw is not None and not np.isnan(pred_hw[i]) else None,
            "mape_naive": mape_naive, "mape_hw": mape_hw,
            "modelo_vencedor": vencedor
        })

df_forecast = spark.createDataFrame(pd.DataFrame(registros))

(df_forecast.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.sop_forecast.fato_forecast_cluster"))

# ── Resumo por loja + cluster ───────────────────────────────
resumo = (
    df_forecast.select("loja_id", "cluster_id", "mape_naive", "mape_hw", "modelo_vencedor")
    .distinct()
    .orderBy("loja_id", "cluster_id")
)
display(resumo)

print(f"Total de séries avaliadas: {resumo.count()}")
print(f"Vitórias Holt-Winters   : {resumo.filter(F.col('modelo_vencedor')=='Holt-Winters').count()}")
print(f"Vitórias Naive Sazonal  : {resumo.filter(F.col('modelo_vencedor')=='Naive Sazonal').count()}")